### Primitive Runnables

##### 1. RunnableSequence

In [31]:
from langchain_core.runnables import RunnableSequence, RunnableParallel, RunnablePassthrough, RunnableLambda, RunnableBranch
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv

In [ ]:
load_dotenv()
prompt = PromptTemplate(
    template="Write down 1 line headline on the {topic}",
    input_variables = ['topic']
)
parser = StrOutputParser()

In [ ]:
model = ChatGoogleGenerativeAI(
    model = "gemini-3.1-flash-lite",
    temperature=0
)

In [ ]:
# RunnableSequence
chain = RunnableSequence(prompt, model, parser)
chain.invoke({'topic': 'Mixture of Expert models'})


'**Mixture of Experts: Scaling AI Intelligence by Activating Only What’s Needed.**'

##### RunnableParallel

In [ ]:
prompt1 = PromptTemplate(
    template="Write a 1 line tweet on the {topic} for my account on X",
    input_variables=['topic']
)
prompt2 = PromptTemplate(
    template="Write a 1 line post on the {topic} for my account on LinkedIn",
    input_variables=['topic']
)

In [ ]:
parallel_chain = RunnableParallel({
    'tweet': RunnableSequence(prompt1, model, parser),
    # we can write like this
    'post' : prompt2 | model | parser
})
parallel_chain.invoke({'topic': 'GANs'})

{'tweet': 'GANs are the ultimate digital artists—teaching machines to dream, create, and blur the line between reality and code. 🎨🤖 #GANs #AI #MachineLearning',
 'post': 'Here are a few options depending on the "vibe" of your profile:\n\n*   **The Visionary:** "GANs aren\'t just generating images; they’re redefining the boundaries of machine creativity."\n*   **The Technical:** "From noise to nuance: Exploring the architecture that powers the most realistic synthetic data in AI."\n*   **The Concise:** "Generative Adversarial Networks: Where two neural networks compete to create perfection."\n*   **The Future-Focused:** "The battle between Generator and Discriminator is fueling the next wave of generative AI innovation."\n\n**Pro-tip:** Add a relevant hashtag like #GenerativeAI #MachineLearning #GANs #DeepLearning to increase reach.'}

##### RunnablePassThrough

- Runnable to passthrough inputs unchanged or with additional keys.

- This Runnable behaves almost like the identity function, except that it can be configured to add additional keys to the output, if the input is a dict.

In [16]:
passthrough = RunnablePassthrough()
passthrough.invoke({'quantity': 2})

{'quantity': 2}

In [22]:
joke_prompt = PromptTemplate(
    template = "Write down a Joke on the {topic}",
    input_variables=['topic']
)
joke_summary_prompt = PromptTemplate(
    template = "Write down the Joke summary in short - {joke}",
    input_variables=['joke']
)

joke_gen_chain = RunnableSequence(joke_prompt, model, parser)
joke_chain = RunnableParallel({
    'joke': RunnablePassthrough(),
    'expalnation': RunnableSequence(joke_summary_prompt, model, parser)
})

In [23]:
final_chain = RunnableSequence(joke_gen_chain, joke_chain)
final_chain.invoke({'topic': 'AI'})

{'joke': "Why did the AI cross the road?\n\nBecause it processed 14 million simulations and determined that was the only way to get to the other side—but it still couldn't tell you which of the pictures contained a traffic light.",
 'expalnation': '**Summary:** The AI used advanced data processing to solve the problem of crossing the road, yet failed at a basic "I\'m not a robot" CAPTCHA test.'}

##### RunnableLambda

In [30]:
def word_count(joke):
    return len(joke.split())

runnable_word_count = RunnableLambda(word_count)

joke_prompt = PromptTemplate(
    template = "Write down a Joke on the {topic}",
    input_variables=['topic']
)

joke_gen_chain = RunnableSequence(joke_prompt, model, parser)
parallel_chain = RunnableParallel({
    'joke': RunnablePassthrough(),
    'word_count': RunnableLambda(lambda x:len(x.split()))
})

joke_chain = RunnableSequence(joke_gen_chain, parallel_chain)
joke_chain.invoke({'topic': 'AI'})


{'joke': "Why did the AI cross the road?\n\nBecause it processed 14 million simulations and determined that was the only way to get to the other side—but it still couldn't tell you which of the pictures contained a traffic light.",
 'word_count': 39}

##### RunnableBranch

In [ ]:
prompt1 = PromptTemplate(
    template="Write down a report on the {topic} in 4-5 lines",
    input_variable=['topic']
)
prompt2 = PromptTemplate(
    template="write down the summary of the - {text} in 1 line.",
    input_variable=['text']
)

report_gen = RunnableSequence(prompt1, model, parser)
conditional_branch = RunnableBranch(
    (lambda x: len(x.split()) > 50, prompt2 | model | parser), #LCEL
    (lambda x: len(x.split()) < 50, RunnableLambda(lambda x: len(x.split()))),
    RunnablePassthrough() #Default
)
joined_chain = RunnableSequence(report_gen, conditional_branch)
joined_chain.invoke({'topic': 'ML'})

'Machine Learning is a subset of AI that uses data-driven algorithms to enable systems to learn, improve, and automate decision-making across various industries.'

##### LangChain Expressive Language (LCEL)

In [41]:
# RunnableSequence(prompt1 , model, parser, prompt2, model, parser)
#                        ||
#                        ||
# chain = prompt1 | model | parser | prompt2 | model | parser